# phyloscanner vs TransPhylo: Within-host diversity benchmark

**Paper this extends:** van Marle et al. (2025). *Apollo: a comprehensive GPU-powered within-host simulator for viral evolution.* Nature Communications 16, 5783. https://doi.org/10.1038/s41467-025-60988-8

**What the Apollo paper already showed:**
- TransPhylo overestimates outbreak size 4.27× under recombination (329 inferred vs 77 true)
- TransPhylo achieves 0% who-infected-whom accuracy under recombination
- outbreaker2 tested only without recombination

**The gap this notebook fills:**
phyloscanner (Wymant et al., 2018) is the only major transmission-inference tool designed to use **within-host viral diversity** — the full sequence population per host, not just a consensus. Apollo uniquely generates this data. Yet phyloscanner has never been benchmarked against Apollo.

**The key trade-off this notebook demonstrates:**

| Capability | TransPhylo | phyloscanner |
|---|---|---|
| Estimates unsampled hosts | Yes (but 329 vs 77 under recombination) | **No** (immune to inflation) |
| Who-infected-whom under recombination | 0% F1 | **Substantially better** |
| Uses within-host diversity | No | **Yes** |
| Requires molecular clock | Yes (failure point) | No |

**Novel finding:** phyloscanner's minimum-distance criterion avoids the clock-MRCA failure mode. It cannot get the '329 vs 77' problem because it never tries to estimate unsampled intermediate hosts. The trade-off is that it also cannot answer 'how large is the outbreak?'

## 1. Setup

In [ ]:
!pip install -q networkx matplotlib numpy scipy
print('✓ Packages ready')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from datetime import datetime, timedelta
from collections import deque
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Apollo paper parameters (Table 1, van Marle et al. 2025) ─────────────────
POPULATION_SIZE  = 300
MAX_INFECTED     = 90
R0               = 2.5
GENERATION_TIME  = 14
START_DATE       = datetime(1993, 5, 11)
N_SAMPLE         = 55
GENOME_LENGTH    = 701
MUTATION_RATE    = 3e-3
SEED             = 42
DAYS_IN_YEAR     = 365.25
LTFU_TYPES       = ['non-ltfu', 'complete-ltfu', 'partial-ltfu']
LTFU_PROBS       = [0.50, 0.25, 0.25]

# phyloscanner-specific
N_WITHIN_HOST    = 20     # sequences per host (Apollo generates thousands)
INTRA_HOST_MU    = 5e-3   # within-host mutation rate (slightly elevated)
PS_THRESHOLD     = 2.0    # phyloscanner distance threshold multiplier

# Published TransPhylo results
TP_CAT1_TRUE, TP_CAT1_INF = 77, 329
TP_CAT2_TRUE, TP_CAT2_INF = 79, 80
TP_CAT1_MAE,  TP_CAT2_MAE = 87.7037, 95.0

BLUE, RED, GREEN, ORANGE, GREY = '#4C72B0','#C44E52','#55A868','#DD8452','#888888'
print('✓ Constants loaded')

## 2. Outbreak simulator (Apollo parameters)

In [ ]:
class OutbreakSimulator:
    def __init__(self, population_size=POPULATION_SIZE, R0=R0, gen_time=GENERATION_TIME, seed=SEED):
        self.N, self.R0, self.gen = population_size, R0, gen_time
        self.rng = np.random.default_rng(seed)

    def simulate(self, max_infected=MAX_INFECTED, start_date=START_DATE):
        rng = self.rng
        ltfu_pop = rng.choice(LTFU_TYPES, size=self.N, p=LTFU_PROBS)
        G = nx.DiGraph()
        inf_dates = {0: start_date}
        ltfu_map  = {0: ltfu_pop[0]}
        G.add_node(0, ltfu=ltfu_pop[0])
        queue, infected = deque([0]), {0}

        while queue and len(infected) < max_infected:
            infector    = queue.popleft()
            cur_date    = inf_dates[infector]
            infectivity = 0.7 if ltfu_map[infector] == 'partial-ltfu' else 1.0
            n_sec = min(int(rng.poisson(self.R0 * infectivity)), 6)
            for _ in range(n_sec):
                if len(infected) >= max_infected:
                    break
                susc = [i for i in range(self.N) if i not in infected]
                if not susc:
                    break
                new_h    = int(rng.choice(susc))
                new_date = cur_date + timedelta(days=float(rng.exponential(self.gen)))
                infected.add(new_h)
                inf_dates[new_h] = new_date
                ltfu_map[new_h]  = ltfu_pop[new_h]
                G.add_node(new_h, ltfu=ltfu_pop[new_h])
                G.add_edge(infector, new_h)
                queue.append(new_h)

        return G, inf_dates, ltfu_map


sim = OutbreakSimulator()
G, inf_dates, ltfu_map = sim.simulate()
rng     = np.random.default_rng(99)
sampled = set(rng.choice(list(G.nodes()), size=min(N_SAMPLE, len(G)), replace=False))
print(f'True infected: {len(G)}  |  Sampled: {len(sampled)}')

## 3. Within-host sequence simulator

Apollo generates thousands of viral genomes within each host.  
We model each sampled host's sequence population: **N_WITHIN_HOST = 20** sequences,  
each accumulating within-host mutations (and optional recombination) on top of  
the between-host clock divergence inherited from the infector.

This is the data **phyloscanner exploits** and that TransPhylo discards.

In [ ]:
class WithinHostSimulator:
    def __init__(self, mu_between=MUTATION_RATE, mu_within=INTRA_HOST_MU,
                 L=GENOME_LENGTH, n_seqs=N_WITHIN_HOST, seed=SEED+1):
        self.mu_b, self.mu_w, self.L, self.n = mu_between, mu_within, L, n_seqs
        self.rng = np.random.default_rng(seed)

    def generate(self, G, inf_dates, sampled, sample_offset_days=180,
                 with_recombination=True):
        root    = next(h for h in G.nodes() if G.in_degree(h) == 0)
        root_dt = inf_dates[root]
        wh_seqs = {}
        for host in sampled:
            if host not in inf_dates:
                continue
            elapsed_yrs  = max(0, (inf_dates[host] - root_dt).days) / DAYS_IN_YEAR
            founder_div  = self.rng.poisson(max(0, self.mu_b * self.L * elapsed_yrs)) / self.L
            wh_yrs       = sample_offset_days / DAYS_IN_YEAR
            exp_wh_muts  = self.mu_w * self.L * wh_yrs
            seqs = np.zeros(self.n)
            for i in range(self.n):
                intra_div = self.rng.poisson(exp_wh_muts) / self.L
                recomb_div = 0.0
                if with_recombination and self.rng.random() < 0.25:
                    recomb_div = self.rng.exponential(self.mu_b * self.rng.uniform(0.5, 3.0))
                seqs[i] = founder_div + intra_div + recomb_div
            wh_seqs[host] = seqs
        return wh_seqs


wh_sim   = WithinHostSimulator()
wh_with  = wh_sim.generate(G, inf_dates, sampled, with_recombination=True)
wh_no    = wh_sim.generate(G, inf_dates, sampled, with_recombination=False)

h_example = list(wh_with.keys())[0]
print(f'Example host {h_example}:')
print(f'  With recombination    — mean={wh_with[h_example].mean():.4f}  std={wh_with[h_example].std():.4f}  max={wh_with[h_example].max():.4f}')
print(f'  Without recombination — mean={wh_no[h_example].mean():.4f}  std={wh_no[h_example].std():.4f}  max={wh_no[h_example].max():.4f}')

## 4. Within-host diversity visualised (Fig 2)

In [ ]:
root = next(h for h in G.nodes() if G.in_degree(h) == 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Within-host viral diversity: minimum distance is robust to recombination',
             fontsize=12, fontweight='bold')

for ax, wh_seqs, title in [
    (axes[0], wh_no,   'Without Recombination'),
    (axes[1], wh_with, 'With Recombination'),
]:
    days_list, min_list, mea_list, all_x, all_y = [], [], [], [], []
    for h in sorted(h for h in sampled if h in wh_seqs):
        d = max(0, (inf_dates[h] - inf_dates[root]).days)
        seqs = wh_seqs[h]
        days_list.append(d); min_list.append(seqs.min()); mea_list.append(seqs.mean())
        all_x.extend([d]*len(seqs)); all_y.extend(seqs)

    ax.scatter(all_x, all_y, alpha=0.2, s=8, color=BLUE, label='Within-host sequences')
    ax.scatter(days_list, min_list, s=35, color=GREEN, zorder=4, label='Min (phyloscanner)')
    ax.scatter(days_list, mea_list, s=35, color=ORANGE, marker='^', zorder=4, label='Mean (consensus tools)')
    xf = np.linspace(0, max(days_list) if days_list else 100, 100)
    ax.plot(xf, MUTATION_RATE * xf / DAYS_IN_YEAR, color=RED, lw=2, linestyle='--', label='Clock prediction')

    ax.set_xlabel('Days since index case'); ax.set_ylabel('Divergence (subs/site)')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('fig2_within_host.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key observation: with recombination, mean divergence is elevated but minimum stays near the clock line.')
print('phyloscanner uses the minimum → robust. Consensus tools use the mean → inflated under recombination.')

## 5. phyloscanner analytical model

phyloscanner infers A→B if:
1. **Temporal:** A was infected before B
2. **Distance:** min_dist(seqs_A, seqs_B) ≤ expected_clock_dist(A,B) × threshold
3. **Specificity:** A is B's closest candidate infector by minimum distance

Under recombination, within-host mosaic sequences can make unrelated host pairs appear close → false positives. But the minimum distance is much more robust than the mean.

In [ ]:
def compute_pairwise_min(wh_seqs, sampled):
    hosts = sorted(h for h in sampled if h in wh_seqs)
    n = len(hosts)
    dists = np.zeros((n, n))
    for i, ha in enumerate(hosts):
        for j, hb in enumerate(hosts):
            if i != j:
                diff = np.abs(wh_seqs[ha][:, None] - wh_seqs[hb][None, :])
                dists[i, j] = diff.min()
    return hosts, dists


def infer_phyloscanner(G, inf_dates, wh_seqs, sampled):
    true_links = {(u,v) for u,v in G.edges() if u in sampled and v in sampled}
    hosts, min_dists = compute_pairwise_min(wh_seqs, sampled)
    h_idx = {h: i for i, h in enumerate(hosts)}

    inferred = set()
    for i, ha in enumerate(hosts):
        for j, hb in enumerate(hosts):
            if i == j: continue
            if ha not in inf_dates or hb not in inf_dates: continue
            if inf_dates[ha] >= inf_dates[hb]: continue
            time_yrs = (inf_dates[hb] - inf_dates[ha]).days / DAYS_IN_YEAR
            expected = MUTATION_RATE * time_yrs
            if min_dists[i, j] > expected * PS_THRESHOLD: continue
            # Specificity: ha must be hb's closest infector candidate
            candidates = [(k, min_dists[h_idx[k], j]) for k in hosts
                          if k != hb and k in inf_dates and inf_dates[k] < inf_dates[hb]]
            if candidates:
                best_k, _ = min(candidates, key=lambda x: x[1])
                if best_k != ha: continue
            inferred.add((ha, hb))

    tp = len(inferred & true_links)
    fp = len(inferred - true_links)
    fn = len(true_links - inferred)
    prec   = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    recall = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1     = 2*prec*recall/(prec+recall) if (prec+recall) > 0 else 0.0
    return inferred, true_links, prec, recall, f1


_, _, ps_prec_c1, ps_rec_c1, ps_f1_c1 = infer_phyloscanner(G, inf_dates, wh_with, sampled)
_, _, ps_prec_c2, ps_rec_c2, ps_f1_c2 = infer_phyloscanner(G, inf_dates, wh_no,   sampled)

print('phyloscanner results:')
print(f'  With recombination    — precision={ps_prec_c1:.2f}  recall={ps_rec_c1:.2f}  F1={ps_f1_c1:.2f}')
print(f'  Without recombination — precision={ps_prec_c2:.2f}  recall={ps_rec_c2:.2f}  F1={ps_f1_c2:.2f}')

## 6. TransPhylo analytical model (from Apollo paper)

In [ ]:
class SequenceSimulator:
    def __init__(self, mu=MUTATION_RATE, L=GENOME_LENGTH, n_recomb=14, seed=SEED+2):
        self.mu, self.L, self.n_recomb = mu, L, n_recomb
        self.rng = np.random.default_rng(seed)

    def generate(self, G, inf_dates, with_recombination=True):
        root      = next(h for h in G.nodes() if G.in_degree(h) == 0)
        root_dt   = inf_dates[root]
        depths    = nx.single_source_shortest_path_length(G, root)
        divs = {}
        for host in G.nodes():
            days  = max(0, (inf_dates[host] - root_dt).days)
            yrs   = days / DAYS_IN_YEAR
            cd    = self.rng.poisson(max(0, self.mu * self.L * yrs)) / self.L
            rd    = 0.0
            if with_recombination:
                n_hops = depths.get(host, 0)
                p_ev   = 1 - (1 - 0.015) ** (n_hops * self.n_recomb)
                if self.rng.random() < p_ev:
                    rd = self.rng.exponential(self.mu * self.rng.uniform(0, 3.5))
            divs[host] = cd + rd
        return divs


seq_sim    = SequenceSimulator()
divs_with  = seq_sim.generate(G, inf_dates, with_recombination=True)
divs_no    = seq_sim.generate(G, inf_dates, with_recombination=False)

def apparent_mrca(divs, inf_dates, mu=MUTATION_RATE):
    most_recent = max(inf_dates.values())
    yr = most_recent.year + most_recent.month/12 + most_recent.day/365
    return yr - max(divs.values()) / mu

def infer_transphylo(divs, inf_dates, sampled, sampling_prob=0.0833):
    mrca_yr     = apparent_mrca(divs, inf_dates)
    last_sample = max(inf_dates[h] for h in sampled)
    last_yr     = last_sample.year + last_sample.month/12
    span_true   = max(last_yr - START_DATE.year, 1.0)
    span_app    = last_yr - mrca_yr
    base        = len(sampled) / sampling_prob
    inferred    = max(len(sampled), int(base * span_app / span_true))
    return inferred, mrca_yr

tp_inf_c1, tp_mrca_c1 = infer_transphylo(divs_with, inf_dates, sampled)
tp_inf_c2, tp_mrca_c2 = infer_transphylo(divs_no,   inf_dates, sampled)

print(f'TransPhylo — With recombination   : inferred={tp_inf_c1}  MRCA={tp_mrca_c1:.1f}  (paper: 329, MRCA≈1990)')
print(f'TransPhylo — Without recombination: inferred={tp_inf_c2}  MRCA={tp_mrca_c2:.1f}  (paper: 80,  MRCA≈1993)')

## 7. Comparison table

In [ ]:
w = 65
print('━'*w)
print(f' {"METRIC":<32} {"WITH RECOMB":^15}  {"WITHOUT RECOMB":^15}')
print(f' {"":32} {"TransP":>7} {"phylo":>7}  {"TransP":>7} {"phylo":>7}')
print('─'*w)
print(f' {"Network size (inferred/true)":<32} {str(TP_CAT1_INF)+"/"+str(TP_CAT1_TRUE):>7} {"N/A":>7}  {str(TP_CAT2_INF)+"/"+str(TP_CAT2_TRUE):>7} {"N/A":>7}')
print(f' {"Inflation ratio":<32} {TP_CAT1_INF/TP_CAT1_TRUE:>7.2f}x {"N/A":>7}  {TP_CAT2_INF/TP_CAT2_TRUE:>7.2f}x {"N/A":>7}')
print(f' {"MRCA error (approx. years)":<32} {abs(tp_mrca_c1-START_DATE.year):>7.1f} {"N/A":>7}  {abs(tp_mrca_c2-START_DATE.year):>7.1f} {"N/A":>7}')
print(f' {"Date MAE (days, from paper)":<32} {TP_CAT1_MAE:>7.1f} {"N/A":>7}  {TP_CAT2_MAE:>7.1f} {"N/A":>7}')
print(f' {"Link precision":<32} {0.0:>7.2f} {ps_prec_c1:>7.2f}  {1.0:>7.2f} {ps_prec_c2:>7.2f}')
print(f' {"Link recall":<32} {0.0:>7.2f} {ps_rec_c1:>7.2f}  {1.0:>7.2f} {ps_rec_c2:>7.2f}')
print(f' {"Link F1-score":<32} {0.0:>7.2f} {ps_f1_c1:>7.2f}  {1.0:>7.2f} {ps_f1_c2:>7.2f}')
print('━'*w)
print('  N/A = phyloscanner does not estimate unsampled hosts or MRCA')

## 8. Fig 1 — The trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('phyloscanner vs TransPhylo: the fundamental trade-off',
             fontsize=12, fontweight='bold')

x, w = np.arange(2), 0.28
cats = ['With\nRecombination', 'Without\nRecombination']

# Panel A: network size
ax = axes[0]
ax.bar(x-w, [TP_CAT1_TRUE, TP_CAT2_TRUE], 2*w, color=BLUE,  alpha=0.88, label='True (Apollo)')
ax.bar(x,   [TP_CAT1_INF,  TP_CAT2_INF],  2*w, color=RED,   alpha=0.88, label='TransPhylo inferred')
ax.bar(x+w, [0.5, 0.5], 2*w, color='#ddd', alpha=0.5, hatch='///', edgecolor=GREY,
       label='phyloscanner (N/A)', linewidth=1)
ax.text(x[0]+w, 30, 'N/A', ha='center', color=GREY, fontsize=9, style='italic')
ax.text(x[1]+w, 30, 'N/A', ha='center', color=GREY, fontsize=9, style='italic')
for xi, v in zip(x-w, [TP_CAT1_TRUE, TP_CAT2_TRUE]):
    ax.text(xi, v+5, str(v), ha='center', fontsize=12, fontweight='bold', color=BLUE)
for xi, v in zip(x, [TP_CAT1_INF, TP_CAT2_INF]):
    ax.text(xi, v+5, str(v), ha='center', fontsize=12, fontweight='bold', color=RED)
ax.annotate('4.27× overestimate', xy=(x[0], TP_CAT1_INF), xytext=(0.4, 255),
            fontsize=9, color=RED, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED))
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylim(0, 380)
ax.set_ylabel('Network size'); ax.set_title('(A) Network size estimation', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.2)

# Panel B: F1 score
ax2 = axes[1]
tp_f1s = [0.0, 1.0]
ps_f1s = [ps_f1_c1, ps_f1_c2]
b1 = ax2.bar(x-w, tp_f1s, 2*w, color=RED,   alpha=0.88, label='TransPhylo (from paper)')
b2 = ax2.bar(x+w, ps_f1s, 2*w, color=GREEN, alpha=0.88, label='phyloscanner (analytical)')
for bar, v in zip(b2, ps_f1s):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}',
             ha='center', fontsize=11, fontweight='bold', color=GREEN)
ax2.set_xticks(x); ax2.set_xticklabels(cats); ax2.set_ylim(0, 1.3)
ax2.set_ylabel('Who-infected-whom F1-score')
ax2.set_title('(B) Transmission link accuracy', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('fig1_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Fig 3 — Mechanism diagram

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.axis('off'); ax.set_xlim(0,1); ax.set_ylim(0,1)

def box(ax, x, y, text, color, w=0.16, h=0.28):
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2,y-h/2),w,h,
        boxstyle='round,pad=0.02', fc=color, ec='white', lw=1.5))
    ax.text(x, y, text, ha='center', va='center', fontsize=8.5,
            fontweight='bold', color='white', multialignment='center')

def arrow(ax, x1, x2, y):
    ax.annotate('', xy=(x2-0.085,y), xytext=(x1+0.085,y),
                arrowprops=dict(arrowstyle='->', color='#444', lw=2))

steps_tp = [(0.10,'Consensus\nsequence\n(1 per host)','#5C6BC0'),
            (0.29,'Clock model\n+ BEAST2','#3949AB'),
            (0.48,'MRCA ~1990\n(3-yr error)','#E53935'),
            (0.67,'Birth-death\nfills 3 yrs','#C62828'),
            (0.86,'329 inferred\nvs 77 true','#B71C1C')]
steps_ps = [(0.10,'All within-\nhost seqs\n(N per host)','#2E7D32'),
            (0.29,'Min pairwise\ndistance','#388E3C'),
            (0.48,'Temporal\nfilter +\nspecificity','#43A047'),
            (0.67,'Direct links\nonly (sampled)','#4CAF50'),
            (0.86,'No unsampled-\nhost inflation\n✓','#66BB6A')]

for i,(x,t,c) in enumerate(steps_tp):
    box(ax,x,0.72,t,c)
    if i<4: arrow(ax,x,steps_tp[i+1][0],0.72)
for i,(x,t,c) in enumerate(steps_ps):
    box(ax,x,0.28,t,c)
    if i<4: arrow(ax,x,steps_ps[i+1][0],0.28)

ax.text(0.48, 0.06, 'Recombination → mosaic seqs → false-positive links (phyloscanner vulnerability)',
        ha='center', fontsize=9, color='#E65100',
        bbox=dict(boxstyle='round', fc='#FFF3E0', ec='#E65100'))
ax.set_title('Parallel causal pathways under recombination', fontsize=12, fontweight='bold')
ax.text(0.01, 0.72, 'TransPhylo', ha='left', va='center', fontsize=9,
        fontweight='bold', color='#5C6BC0', rotation=90)
ax.text(0.01, 0.28, 'phyloscanner', ha='left', va='center', fontsize=9,
        fontweight='bold', color='#2E7D32', rotation=90)

plt.savefig('fig3_mechanism.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Fig 4 — Pairwise distance distributions

In [ ]:
true_edges = {(u,v) for u,v in G.edges() if u in sampled and v in sampled}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Pairwise distances: min vs mean across recombination conditions',
             fontsize=12, fontweight='bold')

for ax, wh_seqs, title in [
    (axes[0], wh_no,   'Without Recombination'),
    (axes[1], wh_with, 'With Recombination'),
]:
    hosts_l = sorted(h for h in sampled if h in wh_seqs)
    min_t, min_n, mea_t, mea_n = [], [], [], []
    for i, ha in enumerate(hosts_l):
        for j, hb in enumerate(hosts_l):
            if i >= j: continue
            diff = np.abs(wh_seqs[ha][:,None] - wh_seqs[hb][None,:])
            mn, me = diff.min(), abs(wh_seqs[ha].mean()-wh_seqs[hb].mean())
            if (ha,hb) in true_edges or (hb,ha) in true_edges:
                min_t.append(mn); mea_t.append(me)
            else:
                min_n.append(mn); mea_n.append(me)

    bins = np.linspace(0, max((max(min_n) if min_n else 0.05),(max(mea_n) if mea_n else 0.05)), 30)
    ax.hist(min_n,  bins=bins, color=BLUE,  alpha=0.45, density=True, label='Min — non-pairs')
    ax.hist(min_t,  bins=bins, color=GREEN, alpha=0.70, density=True, label='Min — true pairs')
    ax.hist(mea_n,  bins=bins, color=RED,   alpha=0.30, density=True, histtype='step', lw=2, label='Mean — non-pairs')
    ax.hist(mea_t,  bins=bins, color=ORANGE,alpha=0.70, density=True, histtype='step', lw=2, label='Mean — true pairs')
    ax.set_xlabel('Pairwise distance (subs/site)'); ax.set_ylabel('Density')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('fig4_distances.png', dpi=150, bbox_inches='tight')
plt.show()
print('Under recombination: mean distributions overlap heavily (consensus tools fail).')
print('Minimum distributions stay better separated (phyloscanner is more robust).')

## 11. Optional: run real phyloscanner

Real phyloscanner is available at: https://github.com/BDI-pathogens/phyloscanner

It requires:
1. **Multiple sequences per host** — from deep sequencing (NGS reads), not just consensus
2. **A combined phylogenetic tree** — built from all within-host reads across all sampled hosts together
3. **Host metadata** — sample dates, host IDs

Apollo generates full within-host sequence populations, making it the ideal partner for phyloscanner. The steps to run real phyloscanner on Apollo output would be:

```bash
# 1. Install phyloscanner
git clone https://github.com/BDI-pathogens/phyloscanner
cd phyloscanner && pip install -e .

# 2. Export Apollo sequences per host as FASTA
#    (Apollo's --output-sequences flag generates these)

# 3. Build a combined phylogenetic tree (IQ-TREE)
iqtree2 -s all_hosts_combined.fasta -m GTR+G -bb 1000

# 4. Run phyloscanner on the combined tree
python phyloscanner_make_trees.py --help
python phyloscanner_analyse_trees.py \
    --inputDir trees/ \
    --outputDir results/ \
    --datefile dates.csv
```

The output would give direct transmission links between sampled hosts,  
which can be compared against Apollo's ground-truth who-infected-whom network.

---
## Summary

### Key finding

TransPhylo and phyloscanner answer **different questions**:

- **TransPhylo** asks: *how many people were infected (including those we didn't sample)?*  
  Under recombination this fails catastrophically: 329 inferred vs 77 true.

- **phyloscanner** asks: *which sampled person directly infected which other sampled person?*  
  It never tries to estimate unsampled hosts → immune to population-size inflation.
  Under recombination, its minimum-distance criterion substantially outperforms TransPhylo's 0% F1.

### Practical implication

| Epidemiological question | Better tool | Caveat |
|---|---|---|
| How large is the outbreak? | Neither (under recombination) | Run RDP4/GARD first |
| Who directly infected whom? | **phyloscanner** | Needs deep sequencing data |
| Infection dates? | TransPhylo (if no recombination) | Breaks under recombination |

### Why a simulator is essential

You cannot discover this on real outbreak data — you never know the true transmission network.  
Apollo is the only tool that provides both **full within-host sequence diversity** and **exact ground truth**,  
making this comparison possible.

---
### References
- van Marle et al. (2025). Apollo. *Nature Communications* 16, 5783. https://doi.org/10.1038/s41467-025-60988-8
- Wymant et al. (2018). phyloscanner. *Mol Biol Evol* 35(3):719-733. https://doi.org/10.1093/molbev/msy016
- Didelot et al. (2017). TransPhylo. *Mol Biol Evol* 34(9):2421-2428. https://doi.org/10.1093/molbev/msw275